# Scanning models for injected code with picklescan and modelscan

> **Not executed by the repo author: run in Colab or in your course lab; versions may need updates.** Check each tool's current documentation and `--help` before trusting a flag or an API name.

**Goal.** Create a pickled "model" that carries a **harmless** payload (it writes a canary file), scan it with real
open-source scanners, and compare with safetensors.

| | |
|---|---|
| Runtime | CPU is enough |
| Time | 15 minutes |
| Offline twin | `labs/model-code-injection-scan/` (a from-scratch scanner with tests; do it first) |

**Safety.** The payload only writes `canary.txt` inside a temporary folder. Never load files from the internet to "see
what happens". Scan first, load only data formats (safetensors), and keep untrusted loads in a sandbox.

In [ ]:
!pip install -q modelscan==0.8.1 picklescan==0.0.21 safetensors==0.4.5 numpy

The versions above are the ones the author believes exist. If pip cannot find one, remove the `==...` part and read the
tool's release notes. The command-line flags below are from memory: run `modelscan --help` and `picklescan --help`.

In [ ]:
import pickle, tempfile, pathlib

work = pathlib.Path(tempfile.mkdtemp(prefix="model-scan-lab-"))
canary = work / "canary.txt"

class HarmlessPayload:
    # __reduce__ tells pickle: "to rebuild me, call this function". A real attacker would name a shell function.
    # We call pathlib.Path.write_text through a helper that only writes our canary.
    def __reduce__(self):
        return (_leave_canary, (str(canary),))

def _leave_canary(path):
    pathlib.Path(path).write_text("CANARY-injected-code-ran")
    return {"layer.weight": [0.0]}

import __main__
__main__._leave_canary = _leave_canary     # make the helper importable by name for pickle

weights = {"layer.weight": [0.5, -0.25, 0.125], "layer.bias": [0.1]}
(work / "clean.pkl").write_bytes(pickle.dumps(weights))
(work / "trapped.pkl").write_bytes(pickle.dumps({**weights, "_extra": HarmlessPayload()}))
print(sorted(p.name for p in work.iterdir()), "| canary exists:", canary.exists())

## Scan with the real tools (nothing is loaded)

In [ ]:
!picklescan --path {work}

In [ ]:
!modelscan -p {work}

Read the reports: both should flag `trapped.pkl` for importing a callable that is not on the safe list, and report
`clean.pkl` as clean. Note what the scanner says about the *name* of the callable. A scanner works from a list of known
risky imports, so a new trick can slip past a denylist. Scanning lowers risk. It does not replace safe formats.

## The unsafe load (do this only here, in a temp folder)

In [ ]:
print("before:", canary.exists())
_ = pickle.loads((work / "trapped.pkl").read_bytes())    # loading IS executing
print("after: ", canary.exists(), "->", canary.read_text())
canary.unlink()

## The safe format: safetensors

In [ ]:
import numpy as np
from safetensors.numpy import save_file, load_file

save_file({"layer.weight": np.array([0.5, -0.25, 0.125], dtype="float32")}, str(work / "model.safetensors"))
print(load_file(str(work / "model.safetensors")))
print("canary exists:", canary.exists())

In [ ]:
!modelscan -p {work}/model.safetensors

## Questions

1. Why can a pickle file run code when loaded, but a safetensors file cannot?
2. Which of these formats are pickle-based in practice: `.pt`, `.pth`, `.bin`, `.ckpt`, `.joblib`, `.npy` with `allow_pickle=True`?
3. A scanner says "clean". What else would you check before loading a model from a stranger? (publisher, signature, hash, license, sandbox)